# 01. Постановка задачи и числовой анализ

Прогнозируется энергопотребление `Usage_kWh` (потребление электроэнергии за 15 минут, кВт·ч) на следующий 15-минутный интервал. Это задача регрессии: каждому интервалу соответствует значение потребления в кВт·ч. Предполагается, что показания завершённого интервала уже доступны к моменту следующего прогноза.

Сначала проверим данные и исследуем связи, которые могут помочь модели, а затем выберем способ подготовки признаков. Графики и их интерпретации собраны в [визуальном анализе](01b_visual_analysis.ipynb).

In [1]:
import pandas as pd
from IPython.display import display
from scipy import stats

from src.data_preparation import TARGET_COLUMN, load_raw_data
from src.features import prepare_feature_dataset

raw_data = load_raw_data()
target = raw_data[TARGET_COLUMN]
feature_dataset = prepare_feature_dataset()
print(raw_data.shape, raw_data.index.min(), raw_data.index.max())
display(raw_data.head())

(35040, 10) 2018-01-01 00:15:00 2019-01-01 00:00:00


,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type
timestamp,,,,,,,,,,
2018-01-01 00:15:00,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load
2018-01-01 00:30:00,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load
2018-01-01 00:45:00,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load
2018-01-01 01:00:00,3.31,3.56,0.0,0.0,68.09,100.0,3600,Weekday,Monday,Light_Load
2018-01-01 01:15:00,3.82,4.50,0.0,0.0,64.72,100.0,4500,Weekday,Monday,Light_Load


При проверке времени обнаружилось, что метка `31/12/2018 00:00` следует за `31/12/2018 23:45`. Поэтому полуночные метки интерпретировали как конец указанных суток и перенесли их на следующий день. После этого получили регулярный временной ряд с шагом 15 минут.

## Аномалии и корреляции

In [2]:
raw_data.corr(numeric_only=True)[TARGET_COLUMN].round(2).sort_values(ascending=False)

Usage_kWh                               1.00
CO2(tCO2)                               0.99
Lagging_Current_Reactive.Power_kVarh    0.90
Lagging_Current_Power_Factor            0.39
Leading_Current_Power_Factor            0.35
NSM                                     0.23
Leading_Current_Reactive_Power_kVarh   -0.32
Name: Usage_kWh, dtype: float64

Расчёт показал сильную корреляцию потребления с CO₂ (0,99) и запаздывающей реактивной энергией (0,90). Однако эти измерения относятся к прогнозируемому интервалу, поэтому использовать их для будущего прогноза нельзя. Связь с временем суток `NSM` (число секунд с начала суток) слабее — 0,23, но линейная корреляция может не отражать более сложную зависимость. Поэтому влияние календаря рассматривается отдельно.

In [3]:
q1, q3 = raw_data[TARGET_COLUMN].quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
print(f"выбросы по IQR (> {upper:.1f} кВт*ч):", (raw_data[TARGET_COLUMN] > upper).sum())
print("максимум:", raw_data[TARGET_COLUMN].max())

выбросы по IQR (> 123.3 кВт*ч): 328
максимум: 157.18


Выше верхней границы межквартильного правила оказались 328 наблюдений; максимум составляет около 157 кВт·ч. Само превышение порога не доказывает ошибку измерения, поэтому сохранили эти значения, чтобы модель учитывала и периоды высокой нагрузки.

In [4]:
day = (pd.DatetimeIndex(raw_data.index) - pd.Timedelta(minutes=15)).normalize()
daily = raw_data.groupby(day).agg(
    usage=(TARGET_COLUMN, "sum"), week_status=("WeekStatus", "first")
)
weekday_median = daily.loc[daily["week_status"] == "Weekday", "usage"].median()
idle_weekdays = daily[
    (daily["week_status"] == "Weekday") & (daily["usage"] < 0.3 * weekday_median)
]
print("медиана суточного потребления в будни:", round(weekday_median))
idle_weekdays.round(0)

медиана суточного потребления в будни: 3119


,usage,week_status
timestamp,,
2018-01-01,352.0,Weekday
2018-02-14,663.0,Weekday
2018-02-15,335.0,Weekday
2018-02-16,332.0,Weekday
2018-05-01,299.0,Weekday
2018-08-01,268.0,Weekday
2018-08-02,276.0,Weekday
2018-08-03,285.0,Weekday
2018-09-24,272.0,Weekday


Следующий шаг — сравнение суточного потребления. В некоторые будни оно составляет менее 30% медианы для будних дней: 1 января, 14–16 февраля, 1 мая, 1–3 августа, 24–26 сентября, 25 и 31 декабря. По имеющимся данным нельзя установить причины снижения. Однако одного признака `WeekStatus` (тип дня: будний или выходной) недостаточно для описания режима работы.

In [5]:
print("нулей:", (target == 0).sum(), " меньше 5 кВт*ч:", f"{(target < 5).mean():.1%}")
print(target.describe().round(2))

нулей: 1  меньше 5 кВт*ч: 53.4%
count    35040.00
mean        27.39
std         33.44
min          0.00
25%          3.20
50%          4.57
75%         51.24
max        157.18
Name: Usage_kWh, dtype: float64


In [6]:
# ошибка 1 кВт*ч в процентах при разных значениях таргета
for true_value in [0.5, 3.0, 60.0]:
    print(true_value, "->", f"{1 / true_value:.0%}")

0.5 -> 200%
3.0 -> 33%
60.0 -> 2%


В распределении встречаются нули, а более половины наблюдений оказались ниже 5 кВт·ч. Нули делают MAPE (средняя абсолютная процентная ошибка) неопределённой, а малые значения усиливают относительную ошибку. Поэтому основной метрикой выбрали MAE (средняя абсолютная ошибка), а дополнительно рассчитываем RMSE (корень из средней квадратичной ошибки) и R² (коэффициент детерминации).

## Признаки

Чтобы учесть недавнюю нагрузку и повторяющиеся режимы, мы добавили к календарным признакам лаги потребления на 1, 96 и 672 интервала — 15 минут, сутки и неделю. Лаги сформированы операцией `shift` (сдвиг значений на заданное число строк); их соответствие прошлым измерениям проверяется отдельно. Первые 672 строки пришлось исключить, поскольку для них ещё нет недельной истории. Подготовленную таблицу сохраняем в `data/processed/steel_features.csv`.

## Гипотезы по признакам

**H1.** Потребление различается по часам суток. Исследуемый признак — `hour` (час суток).

In [7]:
by_hour = feature_dataset.groupby("hour")[TARGET_COLUMN].mean().round(1)
print(by_hour.to_dict())
print(
    stats.kruskal(
        *[group.values for _, group in feature_dataset.groupby("hour")[TARGET_COLUMN]]
    )
)

{0: 7.9, 1: 6.1, 2: 4.4, 3: 4.4, 4: 4.3, 5: 4.2, 6: 4.2, 7: 4.5, 8: 37.7, 9: 58.5, 10: 55.9, 11: 57.0, 12: 18.4, 13: 39.0, 14: 56.1, 15: 55.6, 16: 55.6, 17: 43.5, 18: 33.1, 19: 38.4, 20: 37.7, 21: 13.9, 22: 8.7, 23: 8.1}
KruskalResult(statistic=np.float64(15247.462677915266), pvalue=np.float64(0.0))


Среднее потребление ночью составляет приблизительно 4–8 кВт·ч, а в дневные часы — 40–58 кВт·ч. Около 12 часов наблюдается снижение, а после 21 часа — спад. Такая форма профиля плохо описывается простым линейным ростом с номером часа, поэтому для часа проверяем категориальное представление.

**H2.** Потребление в выходные ниже, чем в будни. Исследуемые признаки — `week_status` (тип дня: будний или выходной) и `day_of_week` (день недели).

In [8]:
print(feature_dataset.groupby("day_of_week")[TARGET_COLUMN].mean().round(1))
weekday = feature_dataset.loc[
    feature_dataset["week_status"] == "Weekday", TARGET_COLUMN
]
weekend = feature_dataset.loc[
    feature_dataset["week_status"] == "Weekend", TARGET_COLUMN
]
print(stats.mannwhitneyu(weekday, weekend))

day_of_week
Friday       32.9
Monday       33.7
Saturday     16.2
Sunday        7.6
Thursday     34.8
Tuesday      34.3
Wednesday    32.2
Name: Usage_kWh, dtype: float64
MannwhitneyuResult(statistic=np.float64(168282223.5), pvalue=np.float64(0.0))


Среднее потребление в будни составило приблизительно 32–35 кВт·ч, в субботу — 16 кВт·ч, а в воскресенье — 8 кВт·ч. Различие наблюдается не только между буднями и выходными, но и между самими выходными днями. Поэтому включили день недели наряду с типом дня.

**H3.** Потребление различается по месяцам. Исследуемый признак — `month` (месяц).

In [9]:
print(feature_dataset.groupby("month")[TARGET_COLUMN].mean().round(1).to_dict())
print(
    stats.kruskal(
        *[group.values for _, group in feature_dataset.groupby("month")[TARGET_COLUMN]]
    )
)

{1: 46.9, 2: 34.0, 3: 27.0, 4: 27.4, 5: 26.6, 6: 22.7, 7: 27.4, 8: 23.0, 9: 20.1, 10: 28.4, 11: 29.9, 12: 20.0}
KruskalResult(statistic=np.float64(3256.023374472351), pvalue=np.float64(0.0))


Сравнение месяцев показало среднее потребление около 47 кВт·ч в январе и 20–23 кВт·ч летом и в сентябре. Однако у нас есть только один год наблюдений, поэтому эти различия нельзя считать доказательством устойчивой годовой сезонности.

**H4.** Потребление связано с предыдущими значениями ряда. Исследуются лаги на 15 минут, сутки и неделю.

In [10]:
feature_dataset[["usage_lag_1", "usage_lag_96", "usage_lag_672"]].corrwith(
    feature_dataset[TARGET_COLUMN]
).round(3)

usage_lag_1      0.913
usage_lag_96     0.608
usage_lag_672    0.677
dtype: float64

Корреляция для лага на 15 минут составила 0,91, для суточного — 0,61, для недельного — 0,68. Более сильная связь с недельным лагом согласуется с различиями между днями недели. Поэтому проверяем все три лага при моделировании: корреляция помогает выбрать кандидатов, а сравнение ошибок покажет, улучшают ли они прогноз.

## Стратегия предобработки

В исходных данных пропусков нет, поэтому заполнение не потребовалось. Неопределённые значения появились только при создании лагов; строки без необходимой истории исключили. Наблюдения выше межквартильного порога сохранили, поскольку ошибки измерений не подтверждены.

Час, день недели и тип дня закодировали методом one-hot (кодирование категорий отдельными бинарными признаками), чтобы модель могла учитывать различия между категориями. Для Ridge сравниваем StandardScaler (стандартизация по среднему и стандартному отклонению), RobustScaler (масштабирование по медиане и межквартильному размаху) и вариант без масштабирования в [экспериментах с базовыми моделями](../modeling/03_baselines.ipynb). Бустинг используем без масштабирования.

Для прогноза нужны сведения, доступные заранее. Поэтому из основных экспериментов исключили: CO₂, реактивные измерения, коэффициенты мощности и `Load_Type` (категория нагрузки). Кодирование и масштабирование обучаем внутри `Pipeline` (цепочка предобработки данных и модели) на обучающей части каждого разбиения, чтобы подготовка признаков не использовала информацию из проверочного периода.

Для разведочной проверки различий используем статистические критерии Краскела — Уоллиса и Манна — Уитни. При этом наблюдения зависят друг от друга во времени, что ограничивает интерпретацию p-values. Поэтому полезность признаков для прогноза проверяем отдельно на временной кросс-валидации.

## Тип нагрузки

Также рассмотрели распределение потребления по категориям `Load_Type` (категория нагрузки), чтобы лучше понять различия режимов. Однако доступность этого признака до начала прогнозируемого интервала не подтверждена, поэтому в модель его не включили.

Обозначения категорий в таблице ниже:

- `Light_Load` — низкая нагрузка.
- `Medium_Load` — средняя нагрузка.
- `Maximum_Load` — максимальная нагрузка.

Это названия категорий в исходных данных; `Maximum_Load` обозначает режим нагрузки, а не единственное максимальное измерение. В таблице `mean` — среднее потребление, `median` — медиана; оба значения указаны в кВт·ч за 15 минут.


In [11]:
display(raw_data.groupby("Load_Type")[TARGET_COLUMN].agg(["mean", "median"]).round(1))

,mean,median
Load_Type,,
Light_Load,8.6,3.3
Maximum_Load,59.3,56.6
Medium_Load,38.4,34.4
